# Movie Rating Prediction

## Setup and Data Loading

First, we'll install and import the necessary libraries. We'll use pandas for data manipulation and numpy for numerical operations. For a dataset, we'll leverage the MovieLens 100k dataset, which is commonly used for movie recommendation and rating prediction tasks. We will download it from a public URL.

In [2]:
import pandas as pd
import numpy as np
import requests
import zipfile
import io

# Download the MovieLens 100k dataset
print("Downloading MovieLens 100k dataset...")
url = "http://files.grouplens.org/datasets/movielens/ml-100k.zip"
response = requests.get(url)
zip_file = zipfile.ZipFile(io.BytesIO(response.content))
zip_file.extractall("ml-100k")
print("Download complete. Files extracted to 'ml-100k/' directory.")

# Load the ratings data
r_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings = pd.read_csv('ml-100k/ml-100k/u.data', sep='\t', names=r_cols, encoding='latin-1')

# Load the movie information
m_cols = ['movie_id', 'title', 'release_date', 'video_release_date', 'imdb_url', 'unknown', 'Action', 'Adventure', 'Animation', 'Children\'s', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
movies = pd.read_csv('ml-100k/ml-100k/u.item', sep='|', names=m_cols, encoding='latin-1')

# Merge the two dataframes
df = pd.merge(ratings, movies, on='movie_id')

print("\nFirst 5 rows of the merged dataset:")
display(df.head())

Download complete. Files extracted to 'ml-100k/' directory.

First 5 rows of the merged dataset:


,user_id,movie_id,rating,timestamp,title,release_date,video_release_date,imdb_url,unknown,Action,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,196,242,3,881250949,Kolya (1996),24-Jan-1997,NaN,http://us.imdb.com/M/title-exact?Kolya%20(1996),0,0,...,0,0,0,0,0,0,0,0,0,0
1,186,302,3,891717742,L.A. Confidential (1997),01-Jan-1997,NaN,http://us.imdb.com/M/title-exact?L%2EA%2E+Conf...,0,0,...,0,1,0,0,1,0,0,1,0,0
2,22,377,1,878887116,Heavyweights (1994),01-Jan-1994,NaN,http://us.imdb.com/M/title-exact?Heavyweights%...,0,0,...,0,0,0,0,0,0,0,0,0,0
3,244,51,2,880606923,Legends of the Fall (1994),01-Jan-1994,NaN,http://us.imdb.com/M/title-exact?Legends%20of%...,0,0,...,0,0,0,0,0,1,0,0,1,1
4,166,346,1,886397596,Jackie Brown (1997),01-Jan-1997,NaN,http://us.imdb.com/M/title-exact?imdb-title-11...,0,0,...,0,0,0,0,0,0,0,0,0,0


## Exploratory Data Analysis (EDA)

Let's start by inspecting the dataframe to understand its structure, check for missing values, and look at some basic statistics.

In [3]:
print("DataFrame Info:")
df.info()

print("\nNumber of unique users:", df['user_id'].nunique())
print("Number of unique movies:", df['movie_id'].nunique())

print("\nRatings distribution:")
display(df['rating'].value_counts().sort_index())

print("\nMissing values per column:")
display(df.isnull().sum()[df.isnull().sum() > 0])

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 27 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   user_id             100000 non-null  int64  
 1   movie_id            100000 non-null  int64  
 2   rating              100000 non-null  int64  
 3   timestamp           100000 non-null  int64  
 4   title               100000 non-null  object 
 5   release_date        99991 non-null   object 
 6   video_release_date  0 non-null       float64
 7   imdb_url            99987 non-null   object 
 8   unknown             100000 non-null  int64  
 9   Action              100000 non-null  int64  
 10  Adventure           100000 non-null  int64  
 11  Animation           100000 non-null  int64  
 12  Children's          100000 non-null  int64  
 13  Comedy              100000 non-null  int64  
 14  Crime               100000 non-null  int64  
 15  Documentary        

,count
rating,
1,6110
2,11370
3,27145
4,34174
5,21201



Missing values per column:


,0
release_date,9
video_release_date,100000
imdb_url,13


## Data Preprocessing

Based on the EDA, we need to perform the following preprocessing steps:
1.  **Drop irrelevant columns**: `video_release_date` has all null values, and `timestamp` is not directly useful for predicting ratings but indicates when the rating was given. `imdb_url` is also likely not a feature for prediction directly.
2.  **Handle `release_date`**: Convert it to a datetime object and address its few missing values.
3.  **Feature Engineering (Implicit)**: Genres are already one-hot encoded, which is good. We might need to consider other features later.

In [4]:
# Drop columns with too many or irrelevant missing values
df_preprocessed = df.drop(columns=['video_release_date', 'timestamp', 'imdb_url', 'title'])

# Handle missing values in 'release_date' by dropping rows (only 9 missing)
df_preprocessed.dropna(subset=['release_date'], inplace=True)

# Convert 'release_date' to datetime objects
df_preprocessed['release_date'] = pd.to_datetime(df_preprocessed['release_date'])

print("DataFrame after dropping columns and handling missing values:")
df_preprocessed.info()

print("\nFirst 5 rows of the preprocessed dataset:")
display(df_preprocessed.head())

DataFrame after dropping columns and handling missing values:
<class 'pandas.core.frame.DataFrame'>
Index: 99991 entries, 0 to 99999
Data columns (total 23 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   user_id       99991 non-null  int64         
 1   movie_id      99991 non-null  int64         
 2   rating        99991 non-null  int64         
 3   release_date  99991 non-null  datetime64[ns]
 4   unknown       99991 non-null  int64         
 5   Action        99991 non-null  int64         
 6   Adventure     99991 non-null  int64         
 7   Animation     99991 non-null  int64         
 8   Children's    99991 non-null  int64         
 9   Comedy        99991 non-null  int64         
 10  Crime         99991 non-null  int64         
 11  Documentary   99991 non-null  int64         
 12  Drama         99991 non-null  int64         
 13  Fantasy       99991 non-null  int64         
 14  Film-Noir     99991 non-null 

,user_id,movie_id,rating,release_date,unknown,Action,Adventure,Animation,Children's,Comedy,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,196,242,3,1997-01-24,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,186,302,3,1997-01-01,0,0,0,0,0,0,...,0,1,0,0,1,0,0,1,0,0
2,22,377,1,1994-01-01,0,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
3,244,51,2,1994-01-01,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,1,1
4,166,346,1,1997-01-01,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Feature Engineering and Model Preparation

We will now prepare the data for our regression model. This includes:
1.  Extracting the release year from the `release_date`.
2.  Defining our features (X) and target (y).
3.  Splitting the data into training and testing sets.

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Feature Engineering: Extract year from release_date
df_preprocessed['release_year'] = df_preprocessed['release_date'].dt.year

# Drop the original release_date column as we have extracted the year
df_preprocessed = df_preprocessed.drop(columns=['release_date'])

# Define features (X) and target (y)
# Features include user_id, movie_id, release_year, and all genre columns
X = df_preprocessed.drop(columns=['rating'])
y = df_preprocessed['rating']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

print("\nFirst 5 rows of X_train:")
display(X_train.head())

Shape of X_train: (79992, 22)
Shape of X_test: (19999, 22)
Shape of y_train: (79992,)
Shape of y_test: (19999,)

First 5 rows of X_train:


,user_id,movie_id,unknown,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,...,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,release_year
38244,527,507,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1951
3808,292,176,0,1,0,0,0,0,0,0,...,0,0,0,0,0,1,1,1,0,1986
27931,535,789,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1995
6008,291,455,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1997
65811,627,197,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1967


## Model Training and Evaluation

We will use a `RandomForestRegressor` for predicting movie ratings. After training, we will evaluate the model's performance using Mean Squared Error (MSE) and R-squared (R2 Score).

In [6]:
# Initialize the RandomForestRegressor model
# Using a small number of estimators (n_estimators) for quicker execution in a demo, can be tuned later
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

print("Training the RandomForestRegressor model...")
# Train the model
model.fit(X_train, y_train)
print("Model training complete.")

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nMean Squared Error (MSE): {mse:.4f}")
print(f"R-squared (R2 Score): {r2:.4f}")

print("\nFirst 10 actual vs predicted ratings:")
results = pd.DataFrame({'Actual': y_test.head(10), 'Predicted': y_pred[:10]})
display(results)

Training the RandomForestRegressor model...
Model training complete.

Mean Squared Error (MSE): 1.2139
R-squared (R2 Score): 0.0420

First 10 actual vs predicted ratings:


,Actual,Predicted
33971,5,3.97
22859,1,2.82
19454,4,3.93
9735,5,4.40
7131,3,2.62
1402,1,3.82
18092,4,3.87
9697,4,3.22
7128,4,3.14
41758,3,3.04
